In [1]:
import pandas as pd
#from pandas.io.parsers import ParserError
import numpy as np
from helper import get_mapper
import json
import os
import re

In [2]:
from os import listdir, stat
from os.path import isfile, join
BASE_DIR = "."
MIN_SIZE = 512

In [3]:
def extract_blockidf(fullname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip()

def extract_blockidf2(fullname, plantname):
    return fullname.split("Generation_DE ")[1].rsplit('[MW]')[0].strip().removeprefix(plantname + " ")

def get_smard_name(f):
    return f.rsplit("/")[2].rsplit("_", 4)[0].strip()

def get_smard_name_win(f):
    f = f.replace("\\","/", 2)
    return f.rsplit("/")[2].rsplit("_", 4)[0].strip()

In [4]:
get_smard_name_win("'.\\2015/Abwinden-Asten_201501010000_201512312359_Stunde_1.csv'")

'Abwinden-Asten'

In [5]:
get_smard_name("./2015/Abwinden-Asten_201501010000_201512312359_hour_1.csv")

'Abwinden-Asten'

In [6]:
def get_files_from_folder(folder):
    onlyfiles = [folder + "/" + f for f in listdir(folder) if isfile(join(folder, f))]
    onlyfiles.sort()
    files = [f for f in onlyfiles if stat(f).st_size > MIN_SIZE]
    return files

In [7]:
def convert2plantid(df, plantname):
    oldcols = list(df.columns)
    newcols = [extract_blockidf2(x, plantname) for x in list(df.columns)[1:]]
    #print(plantname + ":" + str(newcols))
    try:
        newcols2 = ['produced_at'] + [mapper[plantname][x] for x in newcols]
    except KeyError:
        newcols2 = ['produced_at'] + [mapper[plantname][x.split(" ")[-1]] for x in newcols]
    test = dict(zip(oldcols, newcols2))
    result = df.rename(columns=test)
    #print(oldcols)
    #print(newcols2)
    return result

In [8]:
def get_plant_from_prod_name(prodname):
    tmp = ""
    try:
        tmp = mapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    blocklist = tmp['list']
    block = blocklist[0]
    blockid = tmp[block]
    
    try:
        plantidx = bpm.loc[bpm.blockid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [9]:
def get_plant_from_prod_name2(prodname):
    tmp = ""
    try:
        tmp = newmapper[prodname]
    except KeyError:
        #print("plant " + prodname + " not found!")
        return None, None
    seelist = tmp['list']
    block = seelist[0]
    blockid = tmp[block]
    
    try:
        plantidx = seem.loc[seem.sseid == blockid, 'plantid'].item()
    except ValueError:
        return blockid, np.nan
    
    return blockid, plantidx

In [10]:
#bpm = pd.read_csv("../basic/block_plant_mapper.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

/tmp/ipykernel_1935857/2092039524.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [11]:
FOLDERS = [os.path.join(BASE_DIR, o) for o in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR,o))]
FOLDERS.sort()
FOLDERS = FOLDERS[1:-4]
#FOLDERS = FOLDERS[0:1]

In [12]:
FOLDERS

['./2015',
 './2016',
 './2017',
 './2018',
 './2019',
 './2020',
 './2021',
 './2022',
 './2023',
 './2024',
 './2025']

In [13]:
FILES_L = [get_files_from_folder(f) for f in FOLDERS]
FILES = [item for sublist in FILES_L for item in sublist]

In [14]:
#FILES

In [15]:
#mapper = get_mapper('../production/plantmapper.json')
mapper = get_mapper('newestmapper_v2.json')
newmapper = get_mapper('newestmapper_v2.json')

In [16]:
#get_plant_from_prod_name("Buschhaus")

In [17]:
get_plant_from_prod_name2("M_nchen_Nord_2")

(None, None)

In [18]:
#mapper

In [19]:
seem

,plantid,sseid
0,06-02-B10117A007,SEE987197130805
1,06-02-B10117A007,SEE913896693631
2,06-05-100-0030723,BNA1084
3,06-05-100-0431554,BNA0992
4,06-05-100-0431554,BNA0991
...,...,...
1105,SD662-98,SEE927542617698
1106,SD662-98,SEE915847711449
1107,SD662-98,SEE912980761904
1108,SD662-98,SEE918332839635


In [20]:
#FILES

In [21]:
#FILES

In [22]:
def get_year(fn):
    return int(fn.split("/")[1])

In [23]:
def get_year2(fn):
    return int(fn.rsplit("_", 3)[1][0:4])

In [24]:
get_year2("./2015/Boxberg_201501010000_201512312345_Stunde_71.csv")

2015

In [25]:
FILES[0]

'./2015/Dampfkraftwerk_Rheinhafen_201501010000_201512312359_Viertelstunde_1.csv'

In [26]:
for f in FILES:
    #print(f[1])
    fn = get_smard_name_win(f)
    #print(fn)

In [27]:
prod_mapper = []
for f in FILES:
    fn = get_smard_name_win(f)
    f = f.replace("\\","/", 2)
    #print(fn)
    blockid, plantid = get_plant_from_prod_name2(fn)
    #print(fn or "" + ":" + blockid or "" + "->" + plantid or "")
    year = 2015
    try:
        year = get_year2(f)
    except IndexError:
        print(fn)
        pass
    prod_mapper.append([f, blockid, plantid, year])

In [28]:
date_format={'Datum von': '%d-%m-%Y %H:%M'}

In [29]:
df = pd.read_csv("./2025/Kraftwerk_Neurath_202501010000_202512312359_Viertelstunde_58.csv", delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], date_format={'Datum von': '%d-%m-%Y %H:%M'}, on_bad_lines='skip', engine='python')

In [30]:
df

,Datum von,Datum bis,Generation_DE Neurath C [MW] Berechnete Auflösungen,Generation_DE Neurath D [MW] Berechnete Auflösungen,Generation_DE Neurath E [MW] Berechnete Auflösungen,Generation_DE Neurath F [MW] Berechnete Auflösungen,Generation_DE Neurath G [MW] Berechnete Auflösungen,Generation_DE Neurath A [MW] Berechnete Auflösungen,Generation_DE Neurath B [MW] Berechnete Auflösungen,Generation_DE Batteriespeicher Neurath [MW] Berechnete Auflösungen
0,01.01.2025 00:00,01.01.2025 00:15,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
1,01.01.2025 00:15,01.01.2025 00:30,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
2,01.01.2025 00:30,01.01.2025 00:45,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
3,01.01.2025 00:45,01.01.2025 01:00,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
4,01.01.2025 01:00,01.01.2025 01:15,NaN,NaN,NaN,0.000,112.500,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
35051,31.12.2025 22:45,31.12.2025 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35052,31.12.2025 23:00,31.12.2025 23:15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35053,31.12.2025 23:15,31.12.2025 23:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35054,31.12.2025 23:30,31.12.2025 23:45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
def convert2plantid_neurath(df, plantname):
    oldcols = list(df.columns)[0:-1]
    print(oldcols)
    newcols = [extract_blockidf2(x, plantname) for x in list(df.columns)[1:-1]]
    print(newcols)
    #print(plantname + ":" + str(newcols))
    try:
        newcols2 = ['produced_at'] + [mapper[plantname][x] for x in newcols]
    except KeyError:
        newcols2 = ['produced_at'] + [mapper[plantname][x.split(" ")[-1]] for x in newcols]
    test = dict(zip(oldcols, newcols2))
    result = df.rename(columns=test)
    #print(oldcols)
    #print(newcols2)
    return result

In [32]:
#convert2plantid_neurath(df, "Neurath")

In [33]:
#df.dtypes

In [34]:
#df.fillna(0, inplace=True)
#df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [35]:
df

,Datum von,Datum bis,Generation_DE Neurath C [MW] Berechnete Auflösungen,Generation_DE Neurath D [MW] Berechnete Auflösungen,Generation_DE Neurath E [MW] Berechnete Auflösungen,Generation_DE Neurath F [MW] Berechnete Auflösungen,Generation_DE Neurath G [MW] Berechnete Auflösungen,Generation_DE Neurath A [MW] Berechnete Auflösungen,Generation_DE Neurath B [MW] Berechnete Auflösungen,Generation_DE Batteriespeicher Neurath [MW] Berechnete Auflösungen
0,01.01.2025 00:00,01.01.2025 00:15,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
1,01.01.2025 00:15,01.01.2025 00:30,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
2,01.01.2025 00:30,01.01.2025 00:45,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
3,01.01.2025 00:45,01.01.2025 01:00,NaN,NaN,NaN,0.000,113.250,NaN,NaN,NaN
4,01.01.2025 01:00,01.01.2025 01:15,NaN,NaN,NaN,0.000,112.500,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
35051,31.12.2025 22:45,31.12.2025 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35052,31.12.2025 23:00,31.12.2025 23:15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35053,31.12.2025 23:15,31.12.2025 23:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35054,31.12.2025 23:30,31.12.2025 23:45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
prod_mapper_df = pd.DataFrame(prod_mapper)

In [37]:
prod_mapper_df

,0,1,2,3
0,./2015/Dampfkraftwerk_Rheinhafen_201501010000_...,SEE980331579218,BWpf-450-2797933-00000000,2015
1,./2015/Gemeinschaftskraftwerk_St_cken_20150101...,SEE987633796220,NI06060316030,2015
2,./2015/GuD_Dormagen_201501010000_201512312359_...,SEE991220769276,NW300-9002708,2015
3,./2015/GuD_Hamm-Uentrop_201501010000_201512312...,None,None,2015
4,./2015/GuD_Ludwigshafen_Mitte_201501010000_201...,SEE914414538991,RP5000671,2015
...,...,...,...,...
699,./2025/Pumpspeicherkraftwerk_Kopswerk_II_20250...,None,None,2025
700,./2025/Pumpspeicherkraftwerk_Markersbach_20250...,None,None,2025
701,./2025/Pumpspeicherkraftwerk_Vianden_202501010...,None,None,2025
702,./2025/Pumpspeicherkraftwerk_Waldeck_II_202501...,None,None,2025


In [38]:
#df[df.columns[2:]] = df[df.columns[2:]].astype(int)

In [39]:
prd_df = pd.DataFrame(prod_mapper, columns = ['file', 'blockid', 'plantid', 'year'])

In [40]:
dedup = prd_df.dropna(subset=['blockid', 'plantid'])

In [41]:
dedup

,file,blockid,plantid,year
0,./2015/Dampfkraftwerk_Rheinhafen_201501010000_...,SEE980331579218,BWpf-450-2797933-00000000,2015
1,./2015/Gemeinschaftskraftwerk_St_cken_20150101...,SEE987633796220,NI06060316030,2015
2,./2015/GuD_Dormagen_201501010000_201512312359_...,SEE991220769276,NW300-9002708,2015
4,./2015/GuD_Ludwigshafen_Mitte_201501010000_201...,SEE914414538991,RP5000671,2015
6,./2015/GuD_M_nchen_S_d_1_201501010000_20151231...,SEE907223904750,BYS00044,2015
...,...,...,...,...
686,./2025/Kraftwerk_Rostock_202501010000_20251231...,SEE943427876363,MV30000226,2025
688,./2025/Kraftwerk_Schkopau_202501010000_2025123...,SEE947677200282,ST100125,2025
689,./2025/Kraftwerk_Scholven_202501010000_2025123...,SEE952352206145,NW500-0342658,2025
693,./2025/Kraftwerk_Walheim_202501010000_20251231...,SEE964114029633,BWpf-450-2142201-00000000,2025


In [42]:
irschdf = pd.DataFrame()

In [44]:
for index, row in list(dedup.iterrows()):
    dfname = row.iloc[0]
    plantid = str(row.iloc[2])
    year = str(int(row.iloc[3]))
    smardname = get_smard_name(dfname)
    try:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip', engine="python")
        df['Datum von'] = pd.to_datetime(df['Datum von'], format='%d.%m.%Y %H:%M')
        df = df.drop('Datum bis', axis=1)
    except ValueError:
        df = pd.read_csv(dfname, delimiter=";", na_values=['-'], thousands='.', decimal=',', parse_dates=["Datum von"], on_bad_lines='skip', engine="python")
        df = df.drop('Datum bis', axis=1)
    try:
    #print(smardname)            
        newdf = convert2plantid(df, smardname)
        newdf.fillna(0, inplace=True)
        newdf[newdf.columns[1:]] = newdf[newdf.columns[1:]].astype(int)
        newdf.to_csv("./by_plantid/" + year + "/" + plantid + '.csv', index=False)
        if (smardname == 'M_nchen_Nord_2'):
            print('success')
        if (smardname == "Kraftwerk_Lippendorf"):
            irschdf = newdf
    except (ValueError, KeyError, IndexError):
        print(smardname)
        continue
    
    #try:
    #print(dfname, year)
    #print(type(year))
    
    #except TypeError:
    #    print(year, smardname)
    #print(newdf.dtypes)

Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Heizkraftwerk_West
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Heizkraftwerk_West
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Heizkraftwerk_West
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Heizkraftwerk_West
Kraftwerk_Burghausen
Kraftwerk_Emsland
Kraftwerk_Neurath
Kraftwerk_R_merbr_cke
Kraftwerk_Scholven
Kraftwerk_Walsum
Heiz

In [45]:
extract_blockidf('Generation_DE Bergkamen A [MW] Originalauflösungen\n')

'Bergkamen A'

In [46]:
irschdf

,produced_at,SEE966331411864,SEE998199911088
0,01.01.2025 00:00,0,72
1,01.01.2025 00:15,0,72
2,01.01.2025 00:30,0,72
3,01.01.2025 00:45,0,72
4,01.01.2025 01:00,0,72
...,...,...,...
34742,31.12.2025 22:45,0,0
34743,31.12.2025 23:00,0,0
34744,31.12.2025 23:15,0,0
34745,31.12.2025 23:30,0,0
